In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))
os.environ["NANOCHAT_TOKENIZER_DIR"] = "/home/xirui/.cache/nanochat/tokenizer/d10"
import torch
from contextlib import nullcontext
from IPython.display import display
from nanochat.common import autodetect_device_type, compute_init, compute_cleanup
from nanochat.tokenizer import get_tokenizer
from tasks.mmstar import MMStar
from scripts.finevision_cli import load_vlm_checkpoint
from scripts.mmstar_eval import generate_answer, build_visual_tokens

vlm_tag = "d10_finevision_siglippatch_shuf1"
# vlm_tag = "d10_finevision_clippatch_ViT-B-32_pool1"
n_samples = 10
dtype = "bfloat16"
device_type = autodetect_device_type()
ddp, ddp_rank, ddp_local_rank, ddp_world_size, device = compute_init(device_type)
ptdtype = torch.float32 if dtype == "float32" else torch.bfloat16
autocast_ctx = torch.amp.autocast(device_type=device_type, dtype=ptdtype) if device_type == "cuda" else nullcontext()


/home/xirui/nanochat-visual/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-11-30 15:52:46,629 - nanochat.common - INFO - Distributed world size: 1


Autodetected device type: cuda


In [20]:
tokenizer = get_tokenizer()
model, vision = load_vlm_checkpoint(device, vlm_tag=vlm_tag)
model.eval()
vision.eval()


In [21]:
ds = MMStar()
n = min(n_samples, len(ds))
rows = []
with torch.no_grad(), autocast_ctx:
    for idx in range(n):
        ex = ds[idx]
        question = ex["messages"][0]["content"]
        gt = ex["mmstar_answer"]
        image = ex["mmstar_image"]
        visual_tokens = build_visual_tokens(vision, image, device)
        pred = generate_answer(
            model,
            tokenizer,
            question,
            visual_tokens,
            device,
            max_tokens=32,
            temperature=0.0,
            top_k=None,
        )
        rows.append((idx, question, gt, pred, image))

for idx, question, gt, pred, image in rows:
    print(f"#{idx}")
    display(image)
    print(f"Q: {question}")
    print(f"GT: {gt}")
    print(f"Pred: {pred}\n")


In [ ]:
compute_cleanup()